# Step 2 — Data Cleaning

## Purpose
Real-world data is messy. Before a model can learn from it we must:
- Remove duplicate rows (would inflate accuracy artificially)
- Handle missing values (sklearn models cannot accept NaN)
- Normalize text and missing-value markers (`N/A`, `--`, `#DIV/0!`, etc.)
- Drop columns with no signal (constant, all-empty, ID-only)
- Cast booleans to integers (numpy quantile refuses booleans)

This step directly answers: **"What preprocessing steps were used and why?"**

## Project reference
Real implementation: [`../data_loader.py`](../data_loader.py) — function `clean_data()`

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))   # project root from eda/notebooks/

import pandas as pd
import numpy as np
import data_loader as dl

df = dl.generate_synthetic_data(n_rows=2000, seed=42)
print(f'Original shape: {df.shape}')
df.head(3)

## 2.1 Inject realistic dirt for demonstration

We deliberately add common messiness to a clean copy so we can SEE the cleaning steps work.

In [ ]:
dirty = df.copy()

# Inject ~5% missing values into a numeric column
dirty.loc[dirty.sample(100, random_state=1).index, 'unit_price'] = np.nan

# Add an all-empty column
dirty['empty_col'] = np.nan

# Add a constant column (no information)
dirty['constant_col'] = 'always_same'

# Add an Excel artifact column
dirty['Unnamed: 0'] = range(len(dirty))

# Add fake missing-value markers as strings
dirty.loc[dirty.sample(20, random_state=2).index, 'customer_city'] = 'N/A'

print(f'Dirty shape: {dirty.shape}  (added 3 junk columns + missing values)')

## 2.2 Apply the project's `clean_data()` function

This runs all 10 cleaning steps in one call. We'll inspect what happened afterwards.

In [ ]:
clean = dl.clean_data(dirty, target_col='sales_category')
print(f'Cleaned shape: {clean.shape}')
print(f'\nColumns removed: {set(dirty.columns) - set(clean.columns)}')
print(f'Remaining missing values: {clean.isna().sum().sum()}')

## 2.3 What each cleaning step does and WHY

| Step | Action | Why we do it |
|---|---|---|
| Strip whitespace from column names | `" age "` → `"age"` | Prevents `KeyError` later when looking up columns |
| Drop `Unnamed: X` columns | Excel exports the row index as a column | These have no analytical value |
| Drop columns >60% missing | Too sparse to be useful | Imputation would invent fake data |
| Drop ID-like columns (>90% unique) | order_id, uuid, etc. | They identify rows, they don't predict outcomes |
| Drop constant columns | Same value in every row | Provide zero signal to the model |
| Normalize NA markers | `N/A`, `--`, `#DIV/0!` → real NaN | So `isna()` works correctly |
| Coerce string-numerics | `"$1,200.50"` → `1200.50` | Lets the model use the actual numeric meaning |
| Cast booleans → int | `True` → `1` | `numpy.quantile` refuses bool arithmetic |
| Impute numeric NaN with median | Robust to outliers | Most common safe default for numeric ML |
| Impute categorical NaN with mode | The most common value | Avoids creating a fake category |
| Encode categoricals as integers | `"red"` → `0`, `"blue"` → `1` | sklearn models only accept numbers |

## 2.4 Verify the cleaning worked

After cleaning, the DataFrame should have:
- Zero NaN values
- All numeric dtypes
- No constant columns
- No ID columns

In [ ]:
assert clean.isna().sum().sum() == 0, 'Still has NaN'
assert 'empty_col' not in clean.columns, 'empty_col not dropped'
assert 'constant_col' not in clean.columns, 'constant_col not dropped'
assert 'Unnamed: 0' not in clean.columns, 'Unnamed: not dropped'
print('All cleaning assertions passed.')
print(f'\nFinal dtypes:\n{clean.dtypes.value_counts()}')

## Summary of Step 2

From a 19-column dirty DataFrame with 100 missing values and 3 junk columns, the cleaner produced a fully-numeric, fully-populated DataFrame ready for analysis.

**Next:** Step 3 — explore distributions, correlations, and class balance.